# F1 Lap Time Predictor walkthrough

Run the cells from the project root after installing `requirements.txt`. The first cell downloads the configured season if it is not already cached.

In [ ]:
import logging
import pandas as pd
from src.config import CLEAN_PARQUET, RAW_PARQUET
from src.ingest import load_season
from src.clean import clean_laps, finalize
from src.train import train_model

logging.basicConfig(level=logging.INFO)

In [ ]:
raw = load_season()
RAW_PARQUET.parent.mkdir(parents=True, exist_ok=True)
raw.to_parquet(RAW_PARQUET, index=False)
print(f"Loaded {len(raw):,} laps across {raw['Round'].nunique()} rounds")

In [ ]:
clean = finalize(clean_laps(raw))
CLEAN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(CLEAN_PARQUET, index=False)
clean.groupby(['Year', 'Round']).size().describe()

In [ ]:
model = train_model(clean)
print(f"Trained model: {model}")

In [ ]:
import pickle
import shap
from src.config import ENCODER_PATH
from src.explain import generate_shap

with ENCODER_PATH.open('rb') as file:
    encoders = pickle.load(file)
explainer, shap_values, feature_rows = generate_shap(model, clean, encoders)
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.waterfall(shap_values[0])